# King County Housing — Exploratory Data Analysis (EDA)

This notebook explores the King County housing dataset (`kc_house_sales_joined.csv`).

**Objectives:**
- Understand the structure and quality of the data  
- Explore key distributions and relationships  
- Test business-relevant hypotheses  
- Provide actionable recommendations for a non-technical client

**EDA structure:**
1. Understanding the Data  
2. Hypotheses  
3. Explore (Univariate & Data Quality)  
4. Cleaning  
5. Relationships (Bivariate / Multivariate)  
6. Back to the Hypotheses  
7. Fine-tune for Presentation  
8. Insights & Recommendations  


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we are at project root
os.chdir("/Users/keith/GitHub/kc-housing-eda-keith")
PROJECT_ROOT = Path.cwd()
print("Current directory:", PROJECT_ROOT)

# Display settings
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

# Load data
data_path = PROJECT_ROOT / "data" / "kc_house_sales_joined.csv"
df = pd.read_csv(data_path)

df.shape, df.head()

## 1. Understanding the Data

First, inspect structure, column types, and basic descriptive statistics.


In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Hypotheses

Before analysis, we define 4 expectations:

1. **Larger homes (higher `sqft_living`) are more expensive.**  
2. **Higher quality (`grade` / `condition`) → higher price.**  
3. **Some ZIP codes are premium.**  
4. **Newer or recently renovated homes sell for more.**  


## 3. Explore — Missing Values, Duplicates, and Distributions


In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["price"], bins=50)
plt.title("Price Distribution")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(np.log1p(df["price"]), bins=50)
plt.title("Log(1 + Price) Distribution")
plt.xlabel("log(1 + Price)")
plt.ylabel("Count")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df["sqft_living"], bins=40, ax=axes[0])
axes[0].set_title("Living Area (sqft)")

sns.countplot(x=df["bedrooms"], ax=axes[1])
axes[1].set_title("Bedrooms")

sns.countplot(x=df["bathrooms"], ax=axes[2])
axes[2].set_title("Bathrooms")

plt.tight_layout()
plt.show()

## 4. Cleaning


In [ ]:
df = df.drop_duplicates()

# Convert date
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])

# Check numeric ranges
numeric_check_cols = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]
for col in numeric_check_cols:
    if col in df.columns:
        print(col, df[col].min(), df[col].max())

## 5. Relationships — Correlations & Key Drivers


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_cols.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="sqft_living", y="price", alpha=0.3)
plt.title("Living Area vs Price")
plt.xlabel("Living Area (sqft)")
plt.ylabel("Price")
plt.show()

df[["sqft_living", "price"]].corr()

In [ ]:
if "grade" in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x="grade", y="price")
    plt.title("Price by Grade")
    plt.xlabel("Grade")
    plt.ylabel("Price")
    plt.xticks(rotation=45)
    plt.show()

if "condition" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="condition", y="price")
    plt.title("Price by Condition")
    plt.xlabel("Condition")
    plt.ylabel("Price")
    plt.show()

In [ ]:
if "zipcode" in df.columns:
    zip_price = df.groupby("zipcode")["price"].median().sort_values(ascending=False)

if "zipcode" in df.columns:
    top_zips = zip_price.head(15)

    plt.figure(figsize=(12, 5))
    sns.barplot(x=top_zips.index.astype(str), y=top_zips.values)
    plt.title("Top 15 ZIP Codes by Median Price")
    plt.xlabel("ZIP Code")
    plt.ylabel("Median Price")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
if "yr_built" in df.columns:
    df["age_at_sale"] = 2015 - df["yr_built"]

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="age_at_sale", y="price", alpha=0.3)
    plt.title("Age of House vs Price")
    plt.xlabel("Age at Sale (years)")
    plt.ylabel("Price")
    plt.show()

    df[["age_at_sale", "price"]].corr()

if "yr_renovated" in df.columns:
    df["is_renovated"] = (df["yr_renovated"] > 0).astype(int)
    df.groupby("is_renovated")["price"].median()

## 6. Back to the Hypotheses

The analysis supports all four hypotheses:

**1. Larger homes are more expensive.**  
The scatter plot and correlation show a clear positive relationship between living area and price. The effect is strongest up to the mid-range (~2,500–3,000 sqft), after which price increases continue but with diminishing returns.

**2. Higher grade/condition → higher price.**  
The grade and condition boxplots reveal distinct price bands. Higher grades consistently achieve higher medians, while condition has a weaker but still noticeable separation.

**3. Some ZIP codes are clearly premium.**  
The ZIP code median price ranking shows that a small cluster of ZIPs (e.g., 98112, 98004, 98039) sit far above the rest. These areas likely reflect stronger amenities, school districts, and location desirability.

**4. Newer or renovated homes tend to sell for more.**  
Age correlates negatively with price, and homes marked as renovated have higher median prices. Renovation status does not guarantee a premium, but on average it lifts prices.


## 7. Fine-tune – What to Show the Client

For a short client presentation, we should reduce the full analysis to a small set of the strongest, most interpretable visuals. The goal is clarity, not technical depth.

**Recommended visuals:**
1. **Price distribution** – shows the overall market range and the skew.  
2. **Living area vs. price** – demonstrates the clearest and strongest relationship in the dataset.  
3. **Price by grade** – clean segmentation that is easy for non-technical stakeholders to understand.  
4. **Top ZIP code median prices** – highlights location differences and premium neighborhoods.  
5. *(Optional)* Age vs. price – shows the effect of aging or renovation in a simple way.

All other plots generated during EDA (diagnostics, checks, correlations) are important internally but unnecessary for the client-facing version. The final narrative should be concise, visual, and insights-driven.


## 8. Explain – Insights & Recommendations

### Key Insights

1. **Size is a major price driver.**  
   Living area shows a strong positive correlation with price, especially up to a mid-range band. Beyond that, returns per additional square foot diminish.

2. **Quality (grade) and condition segment the market.**  
   Higher-grade properties form clearly higher price bands. There is still some overlap, but median prices step up noticeably with grade.

3. **Premium ZIP codes exist.**  
   A handful of ZIP codes consistently show higher median prices, likely driven by location, schools, amenities, or proximity to key job hubs.

4. **Younger or renovated stock sells at a premium.**  
   Newer or renovated properties tend to achieve higher median prices compared to older, non-renovated homes in the same area.


## 9. Used information & links 

- https://www.unitedstateszipcodes.org/98101/  # detail information about the region and statistical values
